# Test pipeline PDF -> Vector Store

Notebook này giúp quan sát pipeline hiện tại theo từng chặng:

`PDF -> full_pipeline_ingestion -> chunking -> metadata -> hierarchy -> embedding -> VectorStore`

Mặc định dùng `data/FPT_BCTN.pdf` (PDF native). Có thể đổi sang `data/FPT_BCHCD.pdf` để thử luồng PDF scan.

In [1]:
from pathlib import Path
import os
import sys

# Đảm bảo import được package src khi kernel chạy từ bất kỳ thư mục nào.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

Project root: d:\AI AGENT\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies
Python: c:\Users\Lenovo\anaconda3\envs\finagent\python.exe


## 1. Chọn PDF

`run_full_ingestion` định tuyến PDF dựa trên hậu tố tên file:
- `{TICKER}_BCTN.pdf` -> PDF native (`doc_type='native'`)
- `{TICKER}_BCHCD.pdf` -> PDF scan (`doc_type='scan'`)

Không đổi tên tùy ý nếu muốn giữ cơ chế định tuyến hiện tại.

In [2]:
PDF_PATH = PROJECT_ROOT / 'data' / 'FPT_BCTN.pdf'
COLLECTION_NAME = 'notebook_pdf_pipeline_demo'

assert PDF_PATH.exists(), f'Không tìm thấy PDF: {PDF_PATH}'
assert PDF_PATH.suffix.lower() == '.pdf'
print('PDF:', PDF_PATH)
print('Collection:', COLLECTION_NAME)
print('Đổi sang PDF scan bằng: PDF_PATH = PROJECT_ROOT / data / FPT_BCHCD.pdf')

PDF: d:\AI AGENT\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\data\FPT_BCTN.pdf
Collection: notebook_pdf_pipeline_demo
Đổi sang PDF scan bằng: PDF_PATH = PROJECT_ROOT / data / FPT_BCHCD.pdf


## 2. Ingestion: PDF -> SplitDocument

`run_full_ingestion` trả về `list[SplitDocument]`. Mỗi `SplitDocument` chứa `texts`, `tables`, `images` và metadata cấp tài liệu.

In [3]:
from src.ingestion.full_pipeline_ingestion import run_full_ingestion

documents = run_full_ingestion(file_paths=[str(PDF_PATH)])
assert documents, 'Ingestion không tạo SplitDocument nào'

for doc in documents:
    print({
        'ticker': doc.ticker,
        'doc_type': doc.doc_type,
        'report_category': doc.report_category,
        'source_file': doc.source_file,
        'text_pages': len(doc.texts),
        'tables': len(doc.tables),
        'images': len(doc.images),
    })

doc = documents[0]

2026-08-28 00:36:43 | INFO     | src.ingestion.full_pipeline_ingestion:run_full_ingestion:281 - Tìm thấy 1 file để xử lý.
2026-08-28 00:36:43 | INFO     | src.ingestion.full_pipeline_ingestion:run_full_ingestion:337 - [PDF] File d:\AI AGENT\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\data\FPT_BCTN.pdf (theo quy ước tên) → type=native, ticker=FPT, category=bao_cao_thuong_nien
2026-08-28 00:36:45 | INFO     | src.ingestion.full_pipeline_ingestion:_ingest_pdf_native:235 - [NATIVE] Bắt đầu xử lý: d:\AI AGENT\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\data\FPT_BCTN.pdf  (ticker=FPT)
2026-08-28 00:36:45 | INFO     | src.ingestion.native.text_extractor:extract_text_from_pdf:40 - Extracting native text from: FPT_BCTN.pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
2026-08-28 00:36:45 | DEBUG    | src.ingestion.native.pdf_layout:analyze_page:57 - Page 1 layout: 0 tables, 1 images, 0 pur

In [4]:
if doc.texts:
    page = doc.texts[49]
    print('Trang text mẫu:', page.page_num)
    print(page.text[:2000])
    print('layout_info:', page.layout_info)
else:
    print('PDF này không có page text; hãy xem bảng hoặc thử PDF native.')

Trang text mẫu: 50
LÀM CHỦ
CÔNG NGHỆ CHIẾN LƯỢC

CHIẾN LƯỢC PHÁT TRIỂN GIAI ĐOẠN 2026-2028

BÁO CÁO THƯỜNG NIÊN 2025

AI-First

01

THÔNG ĐIỆP BAN LÃNH ĐẠO

Trong kỷ nguyên mới, AI-First là điều kiện tiên quyết 
để doanh nghiệp tồn tại và bứt phá. Dữ liệu từ 
McKinsey cho thấy các công ty dẫn đầu trong tích hợp 
AI toàn diện ghi nhận doanh thu cao hơn 15% và 
hiệu quả vận hành cải thiện 20-25% so với đối thủ. 
Đối với FPT, xu hướng này đòi hỏi sự chuyển dịch từ 
mô hình cung cấp dịch vụ CNTT truyền thống sang 
nền tảng AI-native, trong đó, AI được tích hợp xuyên 
suốt vào toàn bộ giải pháp và quy trình vận hành.

Năm 2025, FPT đã đầu tư mạnh mẽ phát triển hệ sinh thái AI và ứng dụng AI trong hoạt động kinh doanh, 
sản xuất và dịch vụ. Hệ sinh thái AI của FPT dần định hình với nền tảng hạ tầng tính toán hiệu năng cao AI Factory 
tại Việt Nam và Nhật Bản như: FPT.AI, FPT AI Agents, FleziPT….

02

DẤU ẤN 2025

03

TỔNG QUAN VỀ FPT

Trước đó, FPT cũng đã khởi công Trung tâm Trí tuệ nhân tạ

In [9]:
if doc.tables:
    table = doc.tables[0]
    print('Bảng mẫu:', table.title, '| page:', table.page_num)
    print('engine:', table.extraction_engine)
    print('metadata:', table.metadata)
    print(table.markdown[:2000])
else:
    print('Không tìm thấy bảng trong tài liệu này.')

Không tìm thấy bảng trong tài liệu này.


## 3. Chunking: SplitDocument -> flat chunks

Text và table đi qua hai chunker riêng. Bảng lớn hơn threshold sẽ tạo một summary embeddable và một bảng gốc non-embeddable.

In [5]:
from src.chunking import TextChunker, TableChunker

text_chunks = TextChunker().chunk_pages(doc.texts)
table_chunks = TableChunker().chunk_tables(doc.tables)
flat_chunks = text_chunks + table_chunks

print('text_chunks:', len(text_chunks))
print('table_chunks:', len(table_chunks))
print('flat_chunks:', len(flat_chunks))
assert flat_chunks or (not doc.texts and not doc.tables), 'Không tạo được chunk'

for chunk in flat_chunks[:3]:
    print('---')
    print('id:', chunk.chunk_id, '| level:', chunk.level, '| embeddable:', chunk.embeddable)
    print('metadata:', chunk.metadata.model_dump())
    print(chunk.content[:700])

c:\Users\Lenovo\anaconda3\envs\finagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-08-28 00:38:11 | INFO     | src.chunking.text_chunker:__init__:93 - TextChunker khởi tạo: chunk_size=1000, chunk_overlap=200
2026-08-28 00:38:11 | INFO     | src.chunking.text_chunker:chunk_pages:181 - TextChunker: 232 trang → 746 chunks (size=1000, overlap=200)
2026-08-28 00:38:11 | INFO     | src.chunking.table_chunker:__init__:58 - TableChunker khởi tạo: threshold=2000 tokens, model=gemini-3.1-flash-lite
2026-08-28 00:38:11 | INFO     | src.chunking.table_chunker:chunk_tables:100 - TableChunker: 0 bảng → 0 chunks
text_chunks: 746
table_chunks: 0
flat_chunks: 746
---
id: chunk_404422aa9041 | level: 2 | embeddable: True
metadata: {'ticker': None, 'year': None, 'quarter': None, 'report_type': None, 'section': 'TỔNG QUAN VỀ FPT', 'page': 2, 'source_file': None, 'doc_type': None, 'chunk_type': 'text', 'extraction_engine': None}
[TỔNG QUAN VỀ FPT]
LÀM CHỦ
CÔNG NGHỆ CHIẾN LƯỢC

Là tập đoàn công nghệ hàng đầu Việt Nam, FPT kiên định với sứ mệnh tiên phong tạo đột phá, làm chủ công nghệ

In [14]:
from src.chunking import MetadataEnricher

all_chunks = MetadataEnricher().enrich(
    flat_chunks,
    doc_ticker=doc.ticker,
    doc_report_type=doc.report_category,
    doc_type=doc.doc_type,
    doc_source_file=doc.source_file,
    doc_fiscal_year=doc.fiscal_year,
)

for chunk in all_chunks[:3]:
    print(chunk.metadata.model_dump())
assert all(chunk.metadata.ticker == doc.ticker for chunk in all_chunks)

2026-08-28 01:45:15 | INFO     | src.chunking.metadata_enricher:enrich:179 - MetadataEnricher: enriched 0/746 chunks (ticker=FPT, year=None, quarter=None)
{'ticker': 'FPT', 'year': None, 'quarter': None, 'report_type': 'bao_cao_thuong_nien', 'section': 'TỔNG QUAN VỀ FPT', 'page': 2, 'source_file': 'd:\\AI AGENT\\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\\data\\FPT_BCTN.pdf', 'doc_type': 'native', 'chunk_type': 'text', 'extraction_engine': None}
{'ticker': 'FPT', 'year': None, 'quarter': None, 'report_type': 'bao_cao_thuong_nien', 'section': 'BÁO CÁO TÀI CHÍNH HỢP NHẤT', 'page': 3, 'source_file': 'd:\\AI AGENT\\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\\data\\FPT_BCTN.pdf', 'doc_type': 'native', 'chunk_type': 'text', 'extraction_engine': None}
{'ticker': 'FPT', 'year': None, 'quarter': None, 'report_type': 'bao_cao_thuong_nien', 'section': 'LÀM CHỦ\nCÔNG NGHỆ CHIẾN LƯỢC', 'page': 3, 'source_file': 'd:\\AI AGENT\\a-mul

## 4. Hierarchy

`HierarchicalChunker` thêm level 0/1 parent chunks và quan hệ parent-child. Detail chunks vẫn ở level 2.

In [ ]:
from collections import Counter
from src.chunking import HierarchicalChunker

all_chunks, relationships = HierarchicalChunker().build_hierarchy(enriched_chunks)
print('all_chunks:', len(all_chunks))
print('relationships:', len(relationships))
print('levels:', Counter(chunk.level for chunk in all_chunks))
print('embeddable:', Counter(chunk.embeddable for chunk in all_chunks))

for relation in relationships[:3]:
    print(relation.model_dump())

## 5. Embedding và lưu VectorStore

Cell dưới đây mới thực sự gọi Hugging Face API nếu cache chưa có embedding. Cần đặt `HF_TOKEN` trong `.env` hoặc environment của kernel. `EmbeddingPipeline.run()` tự upsert các chunk vào vector store.

In [7]:
from src.database.vector_store import VectorStore

vector_store = VectorStore(collection_name=COLLECTION_NAME)
print('backend:', 'fallback JSON' if vector_store.use_fallback else 'ChromaDB')
print('collection:', vector_store.collection_name)
print('count before:', vector_store.count())

2026-08-28 00:38:37 | INFO     | src.database.vector_store:_init_backend:91 - Initializing ChromaDB Persistent Client at D:\AI AGENT\a-multi-agent-system-for-analyzing-financial-statements-of-Vietnam-companies\data\chroma_db...
2026-08-28 00:38:37 | INFO     | src.database.vector_store:_init_backend:104 - Vector Store collection 'notebook_pdf_pipeline_demo' ready (cosine).
backend: ChromaDB
collection: notebook_pdf_pipeline_demo
count before: 0


In [15]:
from src.chunking import EmbeddingPipeline

embedding_pipeline = EmbeddingPipeline(vector_store=vector_store)
records = embedding_pipeline.run(all_chunks)

print('embedding records:', len(records))
if records:
    print('model:', records[0].model_name)
    print('dimension:', records[0].dim)
print('count after:', vector_store.count())
assert vector_store.count() >= len(records)

2026-08-28 01:46:46 | INFO     | src.chunking.embedding_pipeline:__init__:61 - EmbeddingPipeline (API Mode): model=jina-embeddings-v4, batch_size=32
2026-08-28 01:46:46 | INFO     | src.chunking.embedding_pipeline:run:145 - Embedding 746/746 chunks (skipped 0 non-embeddable)
2026-08-28 01:46:47 | DEBUG    | src.database.embedding_cache:get_batch:113 - Embedding Cache (jina-embeddings-v4): 0 hits, 746 misses out of 746 texts.
2026-08-28 01:46:47 | INFO     | src.chunking.embedding_pipeline:run:156 - Cache: 0 hits, 746 misses
2026-08-28 01:46:47 | INFO     | src.chunking.embedding_pipeline:run:172 -  Đang gọi API cho batch 1-32/746...
2026-08-28 01:46:47 | ERROR    | src.chunking.embedding_pipeline:_call_hf_api:117 - Lỗi mạng khi gọi HF API: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /pipeline/feature-extraction/jina-embeddings-v4 (Caused by NameResolutionError("HTTPSConnection(host='api-inference.huggingface.co', port=443): Failed 

Exception: Đã hết số lần thử (retries) gọi Hugging Face API.

## 6. Kiểm tra và query lại VectorStore

Query trực tiếp bằng embedding để dùng cùng model với lúc index.

In [ ]:
print(vector_store.get_collection_stats())

query = 'Doanh thu và lợi nhuận của công ty trong năm gần nhất'
query_embedding = embedding_pipeline._call_hf_api([query])[0]
results = vector_store.query_by_embedding(query_embedding, n_results=5, where={'ticker': doc.ticker})

for result in results:
    print('---')
    print('similarity:', result.similarity, '| distance:', result.distance)
    print('metadata:', result.metadata)
    print(result.document[:700])

## Troubleshooting

- `HF_TOKEN`: cần cho embedding model mặc định `BAAI/bge-m3`; embedding cell và query cell có gọi network.
- `GOOGLE_API_KEY`: không bắt buộc cho bảng nhỏ; bảng lớn sẽ dùng fallback summary nếu thiếu key.
- PDF scan (`FPT_BCHCD.pdf`): cần các dependency hệ thống cho chuyển PDF thành ảnh và OCR, thường gồm Poppler và Tesseract/PaddleOCR tùy extractor.
- Nếu notebook mở ở thư mục khác, sửa `PROJECT_ROOT` hoặc chạy kernel từ root repository.
- Để thử scan, đổi `PDF_PATH` ở cell cấu hình rồi chạy lại từ đầu.
- Collection notebook dùng tên riêng `notebook_pdf_pipeline_demo`; dữ liệu persistent nằm theo cấu hình trong `configs/database.yaml`.